In [4]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *

In [2]:
def svg_plot(da,savepath,landmask=False,cmap = "RdBu_r",vmin=np.nan,vcenter=np.nan,vmax=np.nan,cbar=False,norm_map=False):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    # --- 2. Set up the projection ---
    central_lon = 210
    proj = ccrs.Robinson(central_longitude=central_lon)

    fig = plt.figure(figsize=(11, 6))
    ax = plt.axes(projection=proj)

    data = da.data
    if norm_map:
        data = data = data/np.std(data)

    # --- 3. Plot the data ---
    # transform=ccrs.PlateCarree() tells cartopy your data is in regular lat/lon coords
    if not np.isnan(vmin):
        mesh = ax.pcolormesh(
                da.lon, da.lat, data,
                transform=ccrs.PlateCarree(),
                cmap=cmap,
                shading="auto",
                norm=mpl.colors.TwoSlopeNorm(vmin=vmin,vcenter=vcenter,vmax=vmax)
            )
    elif not np.isnan(vcenter):
        mag = np.max(np.abs(data))
        mesh = ax.pcolormesh(
            da.lon, da.lat, data,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            norm=mpl.colors.TwoSlopeNorm(vmin=-mag,vcenter=vcenter,vmax=mag),
            shading="auto",
        )
    else:
        mesh = ax.pcolormesh(
            da.lon, da.lat, data,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            shading="auto",
        )

    # --- 4. Add map features ---
    if landmask:
            land = cfeature.NaturalEarthFeature('physical', 'land', '110m', edgecolor='face')
            ax.add_feature(land, facecolor='lightgray')

    ax.coastlines(linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_global()  # ensures full globe is shown

    # Optional: gridlines
    gl = ax.gridlines(draw_labels=False, linewidth=0.3, color="gray", alpha=0.5)

    # # --- 5. Colorbar ---
    if cbar:
        cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.05, shrink=0.7)
        cbar.set_label(r"Spatial variance in pattern ($\sigma$)")

    # ax.set_title(f"Robinson Projection Centered at {central_lon}°E", fontsize=13)

    plt.tight_layout()
    # plt.savefig("robinson_plot.png", dpi=200, bbox_inches="tight")

    plt.title(da.name)

    # Make figure and axes backgrounds transparent
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    # Cartopy adds its own background patch for the map outline — clear that too
    # ax.background_patch.set_alpha(0)   # older cartopy versions
    # or, in newer cartopy versions:
    ax.spines['geo'].set_visible(False)  # optional: remove the border/outline too

    # Save as SVG with transparency
    plt.savefig(savepath, format="svg", transparent=True, bbox_inches="tight")
    # plt.show()

In [3]:
cdo = Cdo()
template_path           = '../amip/data/lowres_template.nc'

In [ ]:
filepath_output_era5     = '/rugenstein-archive/senne/data/obs/ERA5/tas_ERA5_128x64_194001-202512.nc'

In [ ]:
# Remapping for these was done elsewhere
filepath_output_ceres   = '/scratch/leiff/data/obs/CERES/CERES_200303_202604_lowRes.nc'
filepath_output_era5    = '/scratch/leiff/data/obs/ERA5/tas_200303_202512_lowRes.nc'
ERA5    = xc.open_dataset(filepath_output_era5)
CERES   = xc.open_dataset(filepath_output_ceres)